In [4]:
import json
from dotenv import load_dotenv
from openai import OpenAI
import os
load_dotenv()

True

In [20]:
client = OpenAI()

In [21]:
def call_llm(prompt):
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )
        return response.choices[0].message.content.strip()



def load_prompt(filename):
    path = os.path.join('prompt', filename)
    with open(path, 'r') as f:
        return f.read()


In [ ]:
# sample email
test_emails = [
    {
        "id"     : 1,
        "subject": "Can't login — paid for annual plan last week",
        "sender" : "cto@bigclient.com",
        "body"   : "Our entire team can't login. We paid for annual "
                   "plan last week. Board demo in 3 hours.",
        "correct": "Billing",
        "model_output": "Billing"
    },
    {
        "id"     : 2,
        "subject": "App crashes on settings page",
        "sender" : "user@startup.com",
        "body"   : "Every time I open Settings the app crashes. "
                   "Chrome 120, Windows 11.",
        "correct": "Technical"
    },
    {
        "id"     : 3,
        "subject": "Evaluating if we stay on your platform",
        "sender" : "ceo@bigclient.com",
        "body"   : "We have been customers for 2 years. Evaluating competitors. "
                   "Zapier missing, bulk export missing, pricing jumped 40%.",
        "correct": "Billing"
    },
]

email = test_emails[0]
email

{'id': 1,
 'subject': "Can't login — paid for annual plan last week",
 'sender': 'cto@bigclient.com',
 'body': "Our entire team can't login. We paid for annual plan last week. Board demo in 3 hours.",
 'correct': 'Billing'}

In [45]:
template = load_prompt('classify_cot.md')

In [28]:
# print(template)

In [46]:
prompt = template.format(subject=email['subject'], sender=email['sender'], body=email['body'])

In [49]:
result = call_llm(prompt)
# print(result)
# result.split('\n')[1].split('|')[0].strip()
print(result)

THINKING:
1. Surface complaint: Evaluating if we stay on your platform due to missing features and increased pricing.
2. Root cause: Lack of desired integrations (Zapier) and functionality (bulk export), along with a significant price increase.
3. Team that OWNS this: Billing for pricing concerns; Feature Request for missing integrations and functionalities.
4. Business impact: High, as the customer is considering switching to competitors, which could result in lost revenue.

CATEGORY: Feature Request
URGENCY: High


In [41]:
for email in test_emails:
    prompt = template.format(
        subject=email['subject'], sender=email['sender'], body=email['body']
    )
    result = call_llm(prompt)
    email['model_output'] = result.split('\n')[1].split('|')[0].strip()
    
   

In [42]:
test_emails

[{'id': 1,
  'subject': "Can't login — paid for annual plan last week",
  'sender': 'cto@bigclient.com',
  'body': "Our entire team can't login. We paid for annual plan last week. Board demo in 3 hours.",
  'correct': 'Billing',
  'model_output': 'Billing'},
 {'id': 2,
  'subject': 'App crashes on settings page',
  'sender': 'user@startup.com',
  'body': 'Every time I open Settings the app crashes. Chrome 120, Windows 11.',
  'correct': 'Technical',
  'model_output': 'Technical'},
 {'id': 3,
  'subject': 'Evaluating if we stay on your platform',
  'sender': 'ceo@bigclient.com',
  'body': 'We have been customers for 2 years. Evaluating competitors. Zapier missing, bulk export missing, pricing jumped 40%.',
  'correct': 'Billing',
  'model_output': 'Billing'}]

In [70]:
def classify_with_cot(subject,body,sender):
    template = load_prompt('classify_cot.md')
    prompt = template.format(subject=subject, sender=sender, body=body)
    result = call_llm(prompt)

    # extration logic to get category and urgency
    catgeory, urgency = None, None
    for line in result.split('\n'):
        if 'CATEGORY' in line:
            category = line.split(':')[1].strip()
        if 'URGENCY' in line:
            urgency = line.split(':')[1].strip()
    
    return {
        'category': category,
        'urgency': urgency,
        'full_output': result
    }

    

In [52]:
email = test_emails[1]
email

{'id': 2,
 'subject': 'App crashes on settings page',
 'sender': 'user@startup.com',
 'body': 'Every time I open Settings the app crashes. Chrome 120, Windows 11.',
 'correct': 'Technical',
 'model_output': 'Technical'}

In [71]:
output = classify_with_cot(email['subject'], email['body'], email['sender'])

In [82]:
from collections import Counter
data = ['Billing', 'Technical', 'Billing']
Counter(data)


Counter({'Billing': 2, 'Technical': 1})

In [87]:
from collections import Counter
def classify_with_self_consistency(email, n_runs=5):
    template = load_prompt("classify_cot.md")
    results  = []
    traces   = []

    for i in range(n_runs):
        prompt = template.format(
            subject=email["subject"],
            body   =email["body"],
            sender =email["sender"]
        )
        result = call_llm(prompt)
        traces.append(result)

        for line in result.split("\n"):
            if "CATEGORY:" in line:
                results.append(line.replace("CATEGORY:", "").strip())
    votes = Counter(results)
    winner = votes.most_common(1)[0]
    confidence = winner[1] / n_runs *100
    urgency = None
    for line in traces[0].split("\n"):      
          if "URGENCY:" in line:
            urgency = line.replace("URGENCY:", "").strip()
    return {
        "category": winner[0],
        "urgency": urgency,
        "confidence": confidence,
    }

        
        


In [84]:
email

{'id': 2,
 'subject': 'App crashes on settings page',
 'sender': 'user@startup.com',
 'body': 'Every time I open Settings the app crashes. Chrome 120, Windows 11.',
 'correct': 'Technical',
 'model_output': 'Technical'}

In [88]:
classify_with_self_consistency(email)

{'category': 'Technical', 'urgency': 'High', 'confidence': 100.0}

In [ ]:
# result = output.split('\n')
# for i in result:
#     if 'CATEGORY' in i:
#         print(i.split(':')[1].strip())

#     if 



Technical


In [ ]:
def draft_response_with_tot(email, classification):
    template = load_prompt("tree_of_thought.md")
    prompt   = template.format(
        subject =email["subject"],
        body    =email["body"],
        category=classification["category"]
    )
    return call_llm(prompt)